# 第10课 · 点积（dot product）与投影（projection） — a·b = |a||b|cosθ，为什么相似度 = 点积 ÷ 积范数

**目标**：点积 = 逐元素相乘再求和；它能衡量两个向量（vector）“有多像”。

**为什么对 Aurora 重要**：`dft()` 里每个频点 `X[k]` 就是信号数组与复指数序列的点积；注意力机制（attention mechanism）的 QK 打分同理。

← **上一课**　[L09 · 向量代数](L09_vectors.ipynb)

> 上节课学习了 **向量代数**：加法、标量乘法与线性组合，NumPy 实现 + 几何意义。  
> 本课将探讨 **点积与投影**。

## 本课剧情：音乐推荐背后的数学

Spotify 是怎么知道你会喜欢下一首歌的？

它把每首歌描述成一个向量（genre, tempo, energy, ...），然后找出和你听过的歌**方向最接近**的那首。方向的"接近程度"用**点积**来测量。

点积 `a·b = Σaᵢbᵢ`，把两个向量压缩成一个数：
- **正数** → 两个向量大体同向（你喜欢电子音乐，它推类似的）
- **零** → 完全垂直（两首歌毫无共同点）
- **负数** → 相反方向（你喜欢古典，它给了重金属）

但问题来了：一首 10 分钟的长歌，能量数值更大——即使风格很不同，点积也可能很高。

解法：除以两首歌各自的"能量大小"（范数），把点积**归一化**到 [-1, 1]。这就是**余弦相似度**（cosine similarity）——跟向量长度无关，只看方向。

本课实现 `cosine_similarity(a, b)` 和理解正交投影（投影 = 点积 / 模）。

## 1. 点积的定义

两个向量 `a = [a₁, a₂, ..., aₙ]` 和 `b = [b₁, b₂, ..., bₙ]`，点积是：

```
a · b = a₁b₁ + a₂b₂ + ... + aₙbₙ = Σᵢ aᵢbᵢ
```

手算例子：a = [1, 2, 3]，b = [4, 5, 6]

```
1×4 + 2×5 + 3×6 = 4 + 10 + 18 = 32
```

几何解读：`a·b = |a| · |b| · cos(θ)`，其中 θ 是两向量夹角。

这给点积第二种理解方式：**|a| · |b| · cos(θ) = 对长度归一化后的"方向一致性"乘以两者的长度**。

DFT 里，`X[k] = Σ x[n] · e^{-2πikn/N}` 就是信号 x 与第 k 个旋转因子的点积——每个频率分量的"得分"。

## 符号入口：先看形状，再看运算

点积的两个输入都必须是同维向量 `(n,)`，输出是标量。每次遇到新运算，先确认输入的形状匹配，再写计算。

In [ ]:
import numpy as np
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print('逐元素相乘:', a * b)        # [4 10 18]
print('点积(求和):', np.dot(a, b)) # 32
print('也可写作 a @ b =', a @ b)

## 动手观察：两个向量，三种角度关系

运行下面的代码，注意 `np.dot(a, b)` 的正负号如何随两向量夹角变化：方向相同时为正，垂直时为零，反向时为负。这就是点积作为“对齐程度”的直观含义。

In [ ]:
import numpy as np

# 三种角度关系：同向、正交、反向
a = np.array([1., 0.])
cases = [
    ("同向  cosθ = +1", np.array([ 2.,  0.])),
    ("正交  cosθ =  0", np.array([ 0.,  1.])),
    ("反向  cosθ = -1", np.array([-1.,  0.])),
]
print(f"固定向量 a = {a}")
for label, b in cases:
    dot = np.dot(a, b)
    cos = dot / (np.linalg.norm(a) * np.linalg.norm(b))
    print(f"  {label}:  a·b = {dot:+.2f},  cosine = {cos:+.2f}")


> **📌 L12 预告**：下方单元格演示矩阵对向量的缩放变换，属于 L12（矩阵变换）预告内容，现在只需观察形状，L12 将完整推导。


In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：遍历几个向量，观察点积如何响应方向变化

固定向量 `a`，依次与不同方向的 `b` 做点积，观察当 `b` 从同向转到垂直再转到反向时，点积从正数降到零再到负数。

In [ ]:
import numpy as np

# 余弦相似度 = 点积 ÷ 两向量的模之积
pairs = [
    (np.array([1.,0.]), np.array([1.,0.])),   # 完全相同
    (np.array([1.,0.]), np.array([0.,1.])),   # 正交
    (np.array([1.,1.]), np.array([1.,-1.])),  # 正交
    (np.array([3.,4.]), np.array([4.,3.])),   # 锐角
]
for a, b in pairs:
    cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    print(f'cos({a}, {b}) = {cos:+.4f}')


## 2. 余弦相似度：方向有多一致（-1~1，越接近1越像）

`cos = (a·b) / (|a| · |b|)`，其中 `|a|` 是向量长度（下一课细讲）。

## 3. ✏️ 你的任务：实现 `cosine_similarity`

**推理路线**：
1. 公式 `cos = (a·b) / (|a|·|b|)` 把点积除以两个向量的长度之积，消掉大小只留方向——无论向量多长，同向结果始终为 1。
2. 分子是点积 `Σ aᵢbᵢ`，直接用 `np.dot(a, b)` 计算。
3. 分母 `|a|·|b|` 是两个 L2 范数之积（下节推导）；暂时用 `np.linalg.norm()`。
4. 边界验证：同向向量 → 1；垂直向量 → 0；反向向量 → -1。

**参考输入输出**：`a=[1,2,3]`，`b=[4,5,6]` → 点积 = 32，|a|=√14≈3.742，|b|=√77≈8.775，余弦 ≈ **0.974**

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写 `cosine_similarity` 前明确三件事

- 输入：`a`、`b`，形状均为 `(n,)`，维度必须相同
- 关键步骤：分子 `np.dot(a, b)`，分母 `np.linalg.norm(a) * np.linalg.norm(b)`
- 返回：标量，范围 [-1, 1]；垂直向量应得 0，同向应得 1

In [ ]:
def cosine_similarity(a, b):
    """
    返回向量 a 与 b 的余弦相似度，范围 [-1, 1]。
    若任一向量为零向量（模为 0），raise ValueError。
    """
    # ✏️ TODO: 实现余弦相似度
    # 提示：分子 np.dot(a, b)，分母 np.linalg.norm(a) * np.linalg.norm(b)
    # 零向量处理：if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0: raise ValueError("...")
    raise NotImplementedError("请实现 cosine_similarity")


In [ ]:
song1 = np.array([5.0, 0.0, 3.0, 0.0])  # 假装是歌曲特征向量
song2 = np.array([4.0, 0.0, 2.0, 0.0])  # 和 song1 很像
song3 = np.array([0.0, 5.0, 0.0, 4.0])  # 风格不同

try:
    print("sim(1,2) =", round(cosine_similarity(song1, song2), 3), "(应接近 1, 很像)")
    print("sim(1,3) =", round(cosine_similarity(song1, song3), 3), "(应接近 0, 不像)")
    assert abs(cosine_similarity(song1, 2 * song1) - 1.0) < 1e-12, "同向向量余弦相似度应精确 = 1"
    assert cosine_similarity(song1, song2) > 0.9, "同类歌曲相似度应 > 0.9"
    assert abs(cosine_similarity(song1, song3)) < 1e-10, "song1⊥song3 相似度应精确为 0"
    assert cosine_similarity(song1, song2) > cosine_similarity(song1, song3)
    # 边界：零向量应触发 ValueError
    try:
        cosine_similarity(np.zeros(4), song1)
        assert False, "零向量应触发 ValueError"
    except ValueError:
        print("✅ 零向量边界：正确触发 ValueError")
    print("\n✅ 全部通过：你做出了推荐系统的核心度量。")
except NotImplementedError as e:
    print(f"⚠️  尚未实现：{e}\n请在上方 cell 填写 cosine_similarity 函数体。")


**🔗 Aurora 连接**：Music Core 的"猜你喜欢"= 找余弦相似度最高的歌。DFT 里 `X[k]` = 信号 · 第 k 个频率的复指数，也是点积。

## 🎨 图示：外积（→秩1矩阵）

内积 a·b = 标量（下方打印）；外积 a⊗b = 秩1矩阵（下方热力图）。

In [ ]:
import numpy as np
from aurora.laviz import style, vec_times_vec
style()
a_vis, b_vis = [1, 2, 3], [4, 5, 6]
inner = int(sum(x*y for x, y in zip(a_vis, b_vis)))
print(f"内积 a·b = {inner}  （标量，压缩成一个数）")
print("外积 a⊗b = 秩1矩阵（展开成矩阵，见图示）：")
vec_times_vec(a_vis, b_vis);


> **📌 L12 预告**：下方矩阵探针实验展示线性变换对多个向量的效果，完整内容见 L12。


In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：只改一个旋钮

取 `t = np.linspace(0, 1, 1000)`，令 `a = np.sin(2*np.pi*1*t)`（1 Hz），`b = np.sin(2*np.pi*2*t)`（2 Hz）——不同频率。计算 `np.dot(a, b)`，结果接近 0：不同频率的正弦波互相正交。再令 `b = np.sin(2*np.pi*1*t)`（同频率），点积变为正数；令 `b = 2 * np.sin(2*np.pi*1*t)`，点积翻倍——点积正比于振幅乘积。DFT 的 `X[k]` 正是利用这个性质：与第 k 个频率的复指数做点积，匹配频率的幅度最大，其他频率因正交性消去。

## 4. 正交投影：点积的几何应用

**标量投影**（向量 a 在 b 方向上的投影长度）：

$$\text{scalar\_proj}_b(a) = \frac{a \cdot b}{\lvert b \rvert}$$

**向量投影**（向量 a 在 b 方向上的投影向量）：

$$\text{vec\_proj}_b(a) = \frac{a \cdot b}{\lvert b \rvert^2} \cdot b$$

几何含义：把 a 的"影子"投射到 b 的方向上。
- a ⊥ b（正交）→ 投影为零向量
- a ∥ b（同向）→ 投影就是 a 本身
- 最小二乘法的核心：找最优拟合直线，等价于把误差向量投影到最小

**Aurora 连接**：`src/aurora/audio/` 中的频谱分析把信号投影到各正弦/余弦基函数上，即 DFT 的投影解释。


In [ ]:
import numpy as np

def scalar_projection(a, b):
    """a 在 b 方向上的标量投影 = a·b / |b|"""
    return np.dot(a, b) / np.linalg.norm(b)

def vector_projection(a, b):
    """a 在 b 方向上的向量投影 = (a·b / |b|²) * b"""
    return (np.dot(a, b) / np.dot(b, b)) * b

# 示例：a=[3,4] 在 x 轴方向 b=[1,0] 上的投影
a = np.array([3., 4.])
b = np.array([1., 0.])
print(f"a = {a}, b = {b}")
print(f"标量投影（在 b 方向上的长度）= {scalar_projection(a, b):.4f}")  # 应为 3.0
print(f"向量投影（在 b 方向上的向量）= {vector_projection(a, b)}")       # 应为 [3. 0.]

# 验证：正交向量的投影为零
c = np.array([0., 5.])
print(f"\na ⊥ c 时，点积 = {np.dot(a, c)}")
print(f"向量投影 a 在 c 方向 = {vector_projection(a, c)}")  # 应为 [0. 0.]

assert abs(scalar_projection(a, b) - 3.0) < 1e-10, "标量投影应为 3.0"
assert np.allclose(vector_projection(a, b), [3., 0.]), "向量投影应为 [3, 0]"
assert np.allclose(vector_projection(a, c), [0., 0.]), "正交向量投影应为零向量"
print("\n✅ 投影验证通过")


## 本课收束

`cosine_similarity(a, b)` 现在能把 `Σaᵢbᵢ` 归一化到 [-1, 1]。Aurora 的 `dft()` 函数里，每个 `X[k]` 的计算就是信号数组与复指数序列的点积，和本节的代数结构完全相同。下一课：**L11** 讲范数，它是余弦相似度分母的来源，也是衡量向量"大小"的工具。

## ✏️ 白板挑战：点积与余弦相似度手算（目标 8 分钟）

盖上屏幕，纸上作答：

**问 1**：a = [1, 2, 3]，b = [4, 5, 6]，手算 `a·b`（逐元素相乘后求和）。

**问 2**：`cosine_similarity([1, 0], [0, 1])` = ?（两向量垂直，不要用计算器）

**问 3**：`cosine_similarity([3, 0], [6, 0])` = ?（同方向但长度不同）

**问 4**：`scalar_projection([3, 4], [1, 0])` = ?（向量 [3,4] 投影到 x 轴）
公式：`a·b / |b|`，其中 b = [1, 0]。

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：点积
a, b = np.array([1.0, 2.0, 3.0]), np.array([4.0, 5.0, 6.0])
q1 = float(np.dot(a, b))
assert q1 == 32.0, f"期望 32，得到 {q1}"
print(f"Q1 ✅  [1,2,3]·[4,5,6] = 1×4 + 2×5 + 3×6 = {q1}")

# 问2：垂直向量余弦相似度
try:
    q2 = cosine_similarity(np.array([1.0, 0.0]), np.array([0.0, 1.0]))
    assert np.isclose(q2, 0.0, atol=1e-10)
    print(f"Q2 ✅  cos([1,0], [0,1]) = {q2:.4f}（垂直 → 0）")
except NotImplementedError:
    print("Q2 ✅  cos(90°) = 0（cosine_similarity 待实现）")

# 问3：同向不同长度
try:
    q3 = cosine_similarity(np.array([3.0, 0.0]), np.array([6.0, 0.0]))
    assert np.isclose(q3, 1.0, atol=1e-10)
    print(f"Q3 ✅  cos([3,0], [6,0]) = {q3:.4f}（同向 → 1.0，与长度无关）")
except NotImplementedError:
    print("Q3 ✅  同向 → cos = 1.0（cosine_similarity 待实现）")

# 问4：标量投影
a4, b4 = np.array([3.0, 4.0]), np.array([1.0, 0.0])
q4 = float(np.dot(a4, b4)) / float(np.linalg.norm(b4))
assert np.isclose(q4, 3.0, atol=1e-10)
print(f"Q4 ✅  scalar_proj([3,4] → x轴) = {q4}（只取 x 分量）")
print("\n🎉 点积白板挑战通过！余弦相似度 = 方向一致性的量化。")

In [ ]:
# ✏️ 本课自评
l10_review = {
    "cosine_similarity_implemented": None,  # cosine_similarity 实现并通过断言？True/False
    "dot_product_geometry":          None,  # 理解 a·b = |a||b|cosθ 的几何意义？True/False
    "orthogonality_intuition":       None,  # 知道点积=0 ↔ 垂直？True/False
    "whiteboard_passed":             None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l10_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l10_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L10 全部通关！进入 L11：向量范数')

---

→ **下一课**　[L11 · 向量范数](L11_norms.ipynb)

> 下节课将学习 **向量范数**：L1 / L2 / ∞ 范数的计算、几何形状与正则化含义。